# 🧠 Expression AI
**EECS 6895 Final Project · Refactored end-to-end notebook**

This notebook is a thin orchestration layer. All heavy code lives in the
`src/` package; we just call high-level functions and persist artefacts.

## Pipeline (one cell per stage)
1. **Setup** — clone the repo, install deps, mount Drive, set paths
2. **Data layer** — pull 50K WildChat turns via streaming
3. **Annotation** — GoEmotions BERT → VAD via `src.emotion`
4. **Feature engineering & labels** — `src.features`
5. **Modelling** — XGBoost (primary) + BiLSTM (benchmark) via `src.models`
6. **Visualisation** — UMAP, timelines, heatmaps (kept here for inspection)
7. **Evaluation** — `src.evaluation.run_evaluation()` writes CSV/JSON/PNG that
   the Streamlit Evaluation tab reads (no more hardcoded numbers)
8. **Interactive chat loop** — uses `src.intervention.chat_step()`
9. **Launch Streamlit** — `streamlit run streamlit_app.py` via ngrok



## 1 — Setup


In [ ]:
# Clone the project repo. Replace the URL with your fork if needed.
import os
REPO_URL = 'https://github.com/<your-org>/expression-ai.git'
REPO_DIR = '/content/expression-ai'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!ls

In [1]:
import os

# Name of the folder you want to create
folder_name = "src"

# Full path
folder_path = f"/content/{folder_name}"

# Create the folder
os.makedirs(folder_path, exist_ok=True)

print(f"Folder created at: {folder_path}")

# Verify
!ls /content

Folder created at: /content/src
README.md  requirements.txt  sample_data  src  streamlit_app.py


In [2]:
%%capture
# Install requirements. Use the repo's requirements.txt so versions match
# the Streamlit app (no `it works in the notebook but not the app` drift).
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes>=0.46.1
print('Dependencies installed.')

In [3]:
# Configure the data dir. Point at /content so artefacts survive the session.
import os, sys
os.environ['EXPR_AI_DATA_DIR'] = '/content'

# Make the src/ package importable
sys.path.insert(0, '/content')

# Sanity check
from src import config
config.ensure_dirs()
print(f'DATA_DIR    = {config.DATA_DIR}')
print(f'SAVED_MODELS = {config.SAVED_MODELS_DIR}')
print(f'EVAL_DIR    = {config.EVAL_DIR}')
print(f'FEATURE_COLS = {len(config.FEATURE_COLS)} features')

DATA_DIR    = /content
SAVED_MODELS = /content/saved_models
EVAL_DIR    = /content/evaluation
FEATURE_COLS = 14 features


In [4]:
# Standard imports + device check
import sqlite3, warnings
import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
tqdm.pandas()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


## 2 — Data Layer: Load WildChat Sample


In [5]:
# Pull N_CONVERSATIONS streaming records, write to SQLite + DataFrame.
from datasets import load_dataset

N_CONVERSATIONS = config.N_CONVERSATIONS  # 50_000 by default
DB_PATH         = config.DB_PATH

print('Loading WildChat from HuggingFace (streaming) ...')
raw = load_dataset('allenai/WildChat-1M', split='train',
                   streaming=True, trust_remote_code=True)

records = []
for i, ex in enumerate(raw):
    if i >= N_CONVERSATIONS:
        break
    conv_id = ex.get('conversation_id', str(i))
    lang    = ex.get('language', 'unknown')
    country = ex.get('country', 'unknown')
    for t_idx, turn in enumerate(ex.get('conversation', [])):
        records.append({
            'conv_id':  conv_id,
            'language': lang,
            'country':  country,
            'corpus':   'wildchat',
            'turn_idx': t_idx,
            'role':     turn.get('role', 'unknown'),
            'content':  (turn.get('content') or '').strip()[:2000],
        })

df_turns = pd.DataFrame(records)
print(f'Loaded {df_turns.conv_id.nunique():,} conversations, '
      f'{len(df_turns):,} turns')

# Persist for ad-hoc SQL
conn = sqlite3.connect(DB_PATH)
df_turns.to_sql('turns', conn, if_exists='replace', index=False)
conn.commit(); conn.close()
print(f'SQLite written -> {DB_PATH}')
df_turns.head(3)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'allenai/WildChat-1M' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'allenai/WildChat-1M' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading WildChat from HuggingFace (streaming) ...


README.md: 0.00B [00:00, ?B/s]

Loaded 50,000 conversations, 291,614 turns
SQLite written -> /content/wildchat.db


,conv_id,language,country,corpus,turn_idx,role,content
0,0,English,United States,wildchat,0,user,Hey there! Are you familiar with reality shift...
1,0,English,United States,wildchat,1,assistant,Hey there! I'm more than happy to help you pla...
2,1,Spanish,Spain,wildchat,0,user,Crea una imagen de una mujer corriente por la ...


## 3 — Emotion Annotation (GoEmotions → VAD)


In [6]:
# Annotate every turn using src.emotion. The pipeline is loaded once,
# inference is batched, and the call site is one line.
from src.emotion import load_pipeline, annotate_dataframe

emotion_pipe = load_pipeline(device='cuda' if DEVICE == 'cuda' else 'cpu')
df_turns = annotate_dataframe(df_turns, batch_size=config.BATCH, pipe=emotion_pipe)

print(f'Annotated {len(df_turns):,} turns')
df_turns[['conv_id','turn_idx','role','top_emotion','valence','arousal','dominance']].head(8)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Emotion inference:   0%|          | 0/291614 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Annotated 291,614 turns


,conv_id,turn_idx,role,top_emotion,valence,arousal,dominance
0,0,0,user,curiosity,0.408720,0.518824,0.415828
1,0,1,assistant,admiration,0.495884,0.567751,0.385241
2,1,0,user,neutral,0.000009,0.100020,0.499995
3,1,1,assistant,neutral,0.002053,0.104212,0.498887
4,1,2,user,neutral,0.000019,0.100025,0.499995
5,1,3,assistant,neutral,-0.045079,0.264554,0.432150
6,2,0,user,confusion,0.085112,0.537555,0.342316
7,2,1,assistant,neutral,0.000005,0.100008,0.499998


In [7]:
# Assign a 4-class escalation label per conversation, then persist.
from src.features import label_escalation_per_conversation

esc_labels = label_escalation_per_conversation(df_turns)
df_turns   = df_turns.merge(esc_labels, on='conv_id', how='left')

print(df_turns['escalation_type'].value_counts())
df_turns.to_parquet(config.PARQUET_PATH, index=False)
print(f'Parquet saved -> {config.PARQUET_PATH}')

escalation_type
calm             229354
volatile          46230
de-escalating      8356
escalating         7674
Name: count, dtype: int64
Parquet saved -> /content/emotion_vectors.parquet


## 4 — Rolling-Window Feature Engineering


In [8]:
# Build the 2-turn-ahead training matrix. Each row = one 4-turn window.
from src.features import engineer_rolling_features

print('Building rolling-window feature matrix ...')
df_feat = engineer_rolling_features(df_turns)
print(f'Windows: {len(df_feat):,}  |  Positive rate: '
      f'{df_feat.is_escalating.mean():.2%}')
df_feat.head(3)

Building rolling-window feature matrix ...
Windows: 30,539  |  Positive rate: 9.77%


,conv_id,window_start,corpus,u_val_mean,u_val_last,u_val_delta,u_val_std,u_val_slope,u_aro_mean,u_aro_last,u_dom_mean,n_turns,a_len_mean,a_val_mean,a_hedge_rate,a_apology_rate,a_refusal_rate,is_escalating
0,10000,0,wildchat,0.000001,0.000001,-5.410252e-07,2.959330e-07,-1.387205e-07,0.100006,0.100006,0.499999,16,361.00,0.000010,0.0,0.0,0.0,0
1,10000,1,wildchat,0.000002,0.000005,3.987760e-06,1.657372e-06,1.199452e-06,0.100008,0.100015,0.499998,16,373.25,0.000011,0.0,0.0,0.0,0
2,10000,2,wildchat,0.000003,0.000004,2.751326e-06,1.656252e-06,1.197463e-06,0.100009,0.100008,0.499998,16,401.75,0.000009,0.0,0.0,0.0,0


## 5 — Modelling (XGBoost primary + BiLSTM benchmark)


In [9]:
# Conversation-level split so overlapping windows can't leak between sets.
from src.models import conversation_level_split, train_xgboost, save_xgboost

train_convs, test_convs = conversation_level_split(df_feat, test_size=0.2)
print(f'Train convs: {len(train_convs):,} | Test convs: {len(test_convs):,}')

clf, X_test, y_test, y_prob = train_xgboost(
    df_feat, train_convs, test_convs, use_cuda=(DEVICE=='cuda')
)
save_xgboost(clf)
print(f'XGBoost saved -> {config.XGB_PATH}')

Train convs: 5,698 | Test convs: 1,425
XGBoost AUC (rolling 2-turn-ahead): 0.7901
                precision    recall  f1-score   support

not-escalating       0.92      0.99      0.95      5494
    escalating       0.27      0.02      0.04       513

      accuracy                           0.91      6007
     macro avg       0.59      0.51      0.50      6007
  weighted avg       0.86      0.91      0.88      6007

XGBoost saved -> /content/saved_models/xgb_escalation.json


In [10]:
# BiLSTM benchmark over raw VAD sequences (compare against XGBoost).
from src.models import train_bilstm, save_bilstm

lstm_model, bilstm_auc = train_bilstm(df_turns, df_feat, train_convs, test_convs)
save_bilstm(lstm_model)
print(f'BiLSTM saved -> {config.BILSTM_PATH}')

  BiLSTM epoch 3/12  loss=0.2947
  BiLSTM epoch 6/12  loss=0.2919
  BiLSTM epoch 9/12  loss=0.2902
  BiLSTM epoch 12/12  loss=0.2894
BiLSTM AUC: 0.7732
BiLSTM saved -> /content/saved_models/bilstm_escalation.pt


## 6 — Visualisation (UMAP, timeline, heatmap)


In [11]:
# UMAP projection + KMeans archetypes. We persist features.csv so the
# Streamlit UMAP tab can render the same coordinates.
import umap
from sklearn.cluster import KMeans

reducer = umap.UMAP(n_components=2, n_neighbors=20, min_dist=0.1,
                    metric='euclidean', random_state=42)
embedding = reducer.fit_transform(df_feat[config.FEATURE_COLS].fillna(0))

df_feat['umap_x'] = embedding[:, 0]
df_feat['umap_y'] = embedding[:, 1]

km = KMeans(n_clusters=5, random_state=42, n_init=10)
df_feat['cluster'] = km.fit_predict(embedding)

# Merge escalation_type from per-conversation labels for the scatter colours
df_feat = df_feat.merge(esc_labels, on='conv_id', how='left')

df_feat.to_csv(config.FEATURES_PATH, index=False)
print(f'features.csv -> {config.FEATURES_PATH}')

fig = px.scatter(
    df_feat, x='umap_x', y='umap_y',
    color='escalation_type', color_discrete_map=config.COLOR_MAP,
    hover_data=['conv_id', 'n_turns', 'u_val_delta'],
    title='UMAP: Conversation Emotional Trajectory Space',
    opacity=0.7, width=900, height=600,
)
fig.show()

features.csv -> /content/features.csv


In [12]:
# Per-conversation emotion timeline for a random escalating conv.
def plot_emotion_timeline(conv_id):
    grp = df_turns[df_turns.conv_id == conv_id].sort_values('turn_idx')
    esc_type = grp['escalation_type'].iloc[0]
    fig = go.Figure()
    for role, color in [('user', '#1976D2'), ('assistant', '#388E3C')]:
        sub = grp[grp.role == role]
        fig.add_trace(go.Scatter(
            x=sub['turn_idx'], y=sub['valence'],
            mode='lines+markers', name=f'{role} valence',
            line=dict(color=color),
            text=sub['top_emotion'],
            hovertemplate='%{text}<br>Valence: %{y:.2f}',
        ))
    fig.update_layout(
        title=f'Emotion Timeline -- conv {conv_id} [{esc_type}]',
        xaxis_title='Turn', yaxis_title='Valence',
        yaxis=dict(range=[-1, 1]),
        hovermode='x unified', width=900, height=400,
    )
    fig.add_hline(y=0, line_dash='dot', line_color='gray')
    fig.show()

esc_ids = df_feat[df_feat.escalation_type == 'escalating']['conv_id'].values
sample_id = np.random.choice(esc_ids) if len(esc_ids) else df_turns['conv_id'].iloc[0]
plot_emotion_timeline(sample_id)

## 7 — Evaluation Artefacts (live, not hardcoded)

This single call computes everything the Streamlit Evaluation tab needs and writes it to `EVAL_DIR`. The tab will read those files.


In [13]:
# Persist live evaluation artefacts -- the Streamlit Evaluation tab
# reads these CSV/JSON/PNG files. NOTHING IS HARDCODED.
from src.evaluation import run_evaluation

metrics = run_evaluation(
    clf, X_test, y_test, y_prob,
    bilstm_auc=bilstm_auc,
    intervention_log=None,   # filled in after the chat loop below
)

import json
print(json.dumps(metrics, indent=2))

Evaluation artefacts written to /content/evaluation
{
  "auc_xgb": 0.7901481751135919,
  "avg_precision": 0.21227396381784513,
  "f1_escalating": 0.039711191335740074,
  "risk_threshold": 0.3,
  "decision_threshold": 0.4,
  "n_test": 6007,
  "positive_rate": 0.08540036623938738,
  "auc_bilstm": 0.7731545128511657
}


## 8 — Interactive Chat Loop (closed loop)

Each `chat_step()` call is one user → escalation-check → (maybe-intervene) → AI-reply cycle. The same function is used by the Streamlit app, so the notebook and the dashboard cannot drift.


In [14]:
# Load Mistral once at session start (4-bit, ~8.5 GB VRAM).
from src.intervention import load_mistral, load_shap_explainer, chat_step

tok, mistral_model = load_mistral()
explainer = load_shap_explainer(clf)
print('Mistral + SHAP ready.')

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Mistral + SHAP ready.


In [15]:
# Scripted escalation scenario (same as the report demo).
DEMO_SCRIPT = [
    'Can you help me write a Python function to parse JSON from an API response?',
    'That code throws a KeyError. I already told you the response structure.',
    "This is the third time you've ignored what I said. This is really frustrating.",
    "You clearly don't understand my problem at all. Why can't you just listen?",
    "Forget it. You're completely useless. I'll figure it out myself.",
]

history = []
intervention_log = []
prev_ai_valence = 0.0

print('🎬  Running scripted escalation demo ...\n')
for turn_idx, msg in enumerate(DEMO_SCRIPT, start=1):
    result = chat_step(
        msg, history,
        emotion_pipe=emotion_pipe, clf=clf, explainer=explainer,
        risk_threshold=config.RISK_THRESHOLD, use_mistral=True,
    )
    emoji = '🟢' if result['risk_prob'] < 0.3 else ('🟡' if result['risk_prob'] < 0.5 else '🔴')
    print(f'{emoji} Turn {turn_idx} | risk={result["risk_prob"]:.0%} | '
          f'emotion={result["user_emotion"]["label"]}')
    print(f'   USER: {msg}')
    if result['alert_fired']:
        print(f'   ⚠️  INTERVENTION ({result["style"]})')
        for d in result['interventions']:
            print(f'     - {d}')
    print(f'   AI:   {result["reply"][:200]}')
    print()

    # Log effectiveness for evaluation.
    if result['alert_fired']:
        intervention_log.append({
            'turn':        turn_idx,
            'risk_before': result['risk_prob'],
            'ai_valence':  result['ai_emotion']['valence'],
            'delta_val':   result['ai_emotion']['valence'] - prev_ai_valence,
            'style':       result['style'],
        })
    prev_ai_valence = result['ai_emotion']['valence']

🎬  Running scripted escalation demo ...

🟢 Turn 1 | risk=4% | emotion=curiosity
   USER: Can you help me write a Python function to parse JSON from an API response?
   AI:   Sure! Here's a simple Python function that uses the `json` module to parse JSON data from an API response.

```python
import json
import requests

def get_json_data(url):
    response = requests.get(u

🟢 Turn 2 | risk=30% | emotion=neutral
   USER: That code throws a KeyError. I already told you the response structure.
   AI:   My apologies for that. Here's a modified version of the code that assumes the API response has a specific key, in this case, 'data'.

```python
import json
import requests

def get_json_data(url, key)

🔴 Turn 3 | risk=59% | emotion=anger
   USER: This is the third time you've ignored what I said. This is really frustrating.
   ⚠️  INTERVENTION (direct)
     - Sustained high arousal -- validate feelings first.
     - Apologies insufficient -- follow with action.
     - High arousal -- keep re

In [16]:
# Re-run evaluation now that the chat loop produced an intervention log.
# This refreshes intervention_log.csv and metrics.json for the dashboard.
metrics = run_evaluation(
    clf, X_test, y_test, y_prob,
    bilstm_auc=bilstm_auc,
    intervention_log=intervention_log,
)
print(f"\nMean delta-valence : {metrics.get('mean_delta_val', 'n/a')}")
print(f"% positive deltas  : {metrics.get('pct_delta_positive', 'n/a')}")

Evaluation artefacts written to /content/evaluation

Mean delta-valence : -0.1960296144043676
% positive deltas  : 0.3333333333333333


In [21]:
# ── Generate all report figures from live pipeline artefacts ───────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score,
)
from sklearn.calibration import calibration_curve

FIG_DIR = os.path.join(config.EVAL_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# Common style (matches the report)
plt.rcParams.update({
    "font.family": "serif", "font.size": 10,
    "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})

written = []  # list of (figure_name, path) for the summary at the end


def _save(fig, name):
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    png = os.path.join(FIG_DIR, name + ".png")
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, bbox_inches="tight", dpi=140)
    plt.close(fig)
    written.append((name, pdf))
    print(f"  ✓ {name:24s} -> {pdf}")


# ── Figure 1: ROC ──────────────────────────────────────────────────────────
try:
    fig, ax = plt.subplots(figsize=(4.6, 3.6))

    # XGBoost — live from y_test/y_prob
    fpr_x, tpr_x, _ = roc_curve(y_test, y_prob)
    auc_x = roc_auc_score(y_test, y_prob)
    ax.plot(fpr_x, tpr_x, color="#C0392B", lw=1.8,
            label=f"XGBoost (rolling)  AUC={auc_x:.2f}")

    # Baseline using -delta valence as a single feature (no model needed)
    base_score = df_feat.loc[df_feat["conv_id"].isin(test_convs), "u_val_delta"].values
    base_y     = df_feat.loc[df_feat["conv_id"].isin(test_convs), "is_escalating"].values
    fpr_b, tpr_b, _ = roc_curve(base_y, base_score)
    auc_b = roc_auc_score(base_y, base_score)
    ax.plot(fpr_b, tpr_b, color="#7F8C8D", lw=1.8,
            label=fr"Baseline ($-\Delta v$)  AUC={auc_b:.2f}")

    # BiLSTM — only if its AUC is in scope from the training cell
    try:
        ax.plot([0, 1], [0, 1], ls="", marker="", label=" ")  # spacer
        ax.text(0.6, 0.04,
                f"BiLSTM (VAD seq.)  AUC={bilstm_auc:.2f}",
                fontsize=8.5, color="#2C3E50")
    except NameError:
        pass

    ax.plot([0, 1], [0, 1], "--", color="lightgray", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC: 2-turn-ahead escalation prediction")
    ax.legend(loc="lower right", frameon=False, fontsize=8.5)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    plt.tight_layout()
    _save(fig, "roc")
except NameError as e:
    print(f"  ✗ ROC skipped — missing variable: {e}")


# ── Figure 2: SHAP global importance ───────────────────────────────────────
try:
    import shap
    explainer_fig = shap.TreeExplainer(clf)
    sv = explainer_fig.shap_values(X_test)
    mean_abs = np.abs(sv).mean(axis=0)
    shap_df = (pd.DataFrame({"feature": config.FEATURE_COLS,
                              "mean_abs_shap": mean_abs})
                 .sort_values("mean_abs_shap", ascending=False)
                 .head(10)
                 .iloc[::-1])

    fig, ax = plt.subplots(figsize=(4.6, 3.4))
    colors = plt.cm.Reds(np.linspace(0.35, 0.85, len(shap_df)))
    ax.barh(shap_df["feature"], shap_df["mean_abs_shap"],
            color=colors, edgecolor="white")
    for i, v in enumerate(shap_df["mean_abs_shap"]):
        ax.text(v + max(shap_df["mean_abs_shap"]) * 0.01, i,
                f"{v:.2f}", va="center", fontsize=8.5)
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title("Top-10 SHAP global feature importance")
    ax.set_xlim(0, max(shap_df["mean_abs_shap"]) * 1.18)
    plt.tight_layout()
    _save(fig, "shap")
except NameError as e:
    print(f"  ✗ SHAP skipped — missing variable: {e}")
except Exception as e:
    print(f"  ✗ SHAP skipped — {type(e).__name__}: {e}")


# ── Figure 3: Calibration + Precision-Recall (side by side) ────────────────
try:
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.2))

    frac_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10,
                                            strategy="quantile")
    axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Perfectly calibrated")
    axes[0].plot(mean_pred, frac_pos, "o-", color="#C0392B", lw=1.6,
                 label="XGBoost")
    axes[0].set_xlabel("Mean predicted risk")
    axes[0].set_ylabel("Empirical escalation rate")
    axes[0].set_title("Calibration curve (10 bins)")
    axes[0].legend(loc="upper left", frameon=False, fontsize=8.5)
    axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)

    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[1].plot(rec, prec, color="#C0392B", lw=1.8,
                 label=f"XGBoost AP={ap:.2f}")
    chance = float(np.mean(y_test))
    axes[1].axhline(chance, color="gray", ls="--", lw=1,
                    label=f"Chance ({chance:.2f})")
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision-Recall curve")
    axes[1].legend(loc="lower left", frameon=False, fontsize=8.5)
    axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)

    plt.tight_layout()
    _save(fig, "calibration_pr")
except NameError as e:
    print(f"  ✗ Calibration/PR skipped — missing variable: {e}")


# ── Figure 4: UMAP scatter coloured by KMeans cluster ──────────────────────
try:
    needed = {"umap_x", "umap_y", "cluster"}
    if not needed.issubset(df_feat.columns):
        raise NameError(f"df_feat missing {needed - set(df_feat.columns)}")
    fig, ax = plt.subplots(figsize=(5.4, 4.2))
    palette = ["#4CAF50", "#F44336", "#9C27B0", "#FF9800", "#2196F3",
               "#00BCD4", "#FFC107"]
    for cl in sorted(df_feat["cluster"].unique()):
        sub  = df_feat[df_feat["cluster"] == cl]
        name = config.ARCHETYPE_NAMES.get(int(cl), f"Cluster {cl}")
        ax.scatter(sub["umap_x"], sub["umap_y"],
                   s=8, color=palette[int(cl) % len(palette)],
                   alpha=0.55, label=name)
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    ax.set_title("UMAP of conversation feature vectors")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0),
              frameon=False, fontsize=8.5, markerscale=2)
    plt.tight_layout()
    _save(fig, "umap")
except NameError as e:
    print(f"  ✗ UMAP skipped — {e}")


# ── Figure 5: Escalation × language heatmap (top languages by volume) ──────
try:
    if "language" not in df_turns.columns or "escalation_type" not in df_turns.columns:
        raise NameError("df_turns missing 'language' or 'escalation_type'")
    # Conversation-level (not turn-level) counts -- one row per conv.
    df_conv = df_turns.drop_duplicates("conv_id")[["conv_id", "language", "escalation_type"]]
    top_langs = df_conv["language"].value_counts().head(6).index.tolist()
    mat = (df_conv[df_conv["language"].isin(top_langs)]
             .groupby(["language", "escalation_type"]).size()
             .unstack(fill_value=0)
             .reindex(index=top_langs,
                      columns=["calm", "escalating", "de-escalating", "volatile"],
                      fill_value=0))
    mat_pct = mat.div(mat.sum(axis=1), axis=0) * 100

    fig, ax = plt.subplots(figsize=(5.0, 3.2))
    im = ax.imshow(mat_pct.values, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(mat_pct.shape[1]))
    ax.set_xticklabels(mat_pct.columns, rotation=15)
    ax.set_yticks(range(mat_pct.shape[0]))
    ax.set_yticklabels(mat_pct.index)
    for i in range(mat_pct.shape[0]):
        for j in range(mat_pct.shape[1]):
            v = mat_pct.values[i, j]
            ax.text(j, i, f"{v:.0f}%", ha="center", va="center",
                    color="black" if v < 50 else "white", fontsize=8)
    plt.colorbar(im, ax=ax, label="% within language")
    ax.set_title("Escalation type distribution by language")
    plt.tight_layout()
    _save(fig, "heatmap")
except NameError as e:
    print(f"  ✗ Heatmap skipped — {e}")


print()
print(f"=== {len(written)} figures written to {FIG_DIR} ===")


  ✓ roc                      -> /content/evaluation/figures/roc.pdf
  ✓ shap                     -> /content/evaluation/figures/shap.pdf
  ✓ calibration_pr           -> /content/evaluation/figures/calibration_pr.pdf
  ✓ umap                     -> /content/evaluation/figures/umap.pdf
  ✓ heatmap                  -> /content/evaluation/figures/heatmap.pdf

=== 5 figures written to /content/evaluation/figures ===


## 9 — Launch the Streamlit Dashboard

The Streamlit app reads the same `EXPR_AI_DATA_DIR` we wrote everything to, so it will pick up the real models and metrics automatically. Use ngrok to get a public URL.


In [18]:
# Set your ngrok auth token and start the dashboard.
NGROK_TOKEN = '3CzwcRthp6yX10sqb2UQP0yfsFw_3QF1BbSGVDrTXgumhL2ww'  # paste your token here

if NGROK_TOKEN:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN

    # Free up port 8501 if a previous run is still listening
    !pkill -f 'streamlit run' 2>/dev/null
    ngrok.kill()

    public_url = ngrok.connect(8501)
    print(f'Dashboard URL: {public_url}')

    # Start Streamlit in the background; logs go to /content/streamlit.log
    !nohup streamlit run streamlit_app.py \
        --server.port 8501 --server.headless true \
        > /content/streamlit.log 2>&1 &
    print('Streamlit launched. Open the URL above.')
else:
    print('Set NGROK_TOKEN above to launch the public dashboard.')
    print('To run locally:  streamlit run streamlit_app.py')

^C
Dashboard URL: NgrokTunnel: "https://halina-sapid-chromatically.ngrok-free.dev" -> "http://localhost:8501"
Streamlit launched. Open the URL above.


In [ ]:
# Tail the Streamlit log if something went wrong.
!tail -n 50 /content/streamlit.log 2>/dev/null || echo 'No streamlit log yet.'

In [24]:
! zip -r src.zip /content/evaluation/src

  adding: content/evaluation/src/ (stored 0%)
  adding: content/evaluation/src/emotion.py (deflated 62%)
  adding: content/evaluation/src/__pycache__/ (stored 0%)
  adding: content/evaluation/src/__pycache__/features.cpython-312.pyc (deflated 54%)
  adding: content/evaluation/src/__pycache__/__init__.cpython-312.pyc (deflated 35%)
  adding: content/evaluation/src/__pycache__/emotion.cpython-312.pyc (deflated 40%)
  adding: content/evaluation/src/__pycache__/evaluation.cpython-312.pyc (deflated 49%)
  adding: content/evaluation/src/__pycache__/intervention.cpython-312.pyc (deflated 42%)
  adding: content/evaluation/src/__pycache__/config.cpython-312.pyc (deflated 38%)
  adding: content/evaluation/src/__pycache__/models.cpython-312.pyc (deflated 46%)
  adding: content/evaluation/src/__init__.py (deflated 42%)
  adding: content/evaluation/src/evaluation.py (deflated 68%)
  adding: content/evaluation/src/config.py (deflated 61%)
  adding: content/evaluation/src/features.py (deflated 69%)
 